# 外部ライブラリなしで強化学習を始める: 最小のQ学習ループ

`jpoke.rl` は行動空間の列挙・合法手マスク・報酬シェーピング・`reset()`/`step()`
形式の環境ラッパー（`RLBattleEnv`）を、`gymnasium` 等の外部ライブラリ抜きで提供する。
本サンプルは、それらの上に標準ライブラリのみ（`random`/`collections.defaultdict`）で
実装した最小のテーブル型Q学習を組み、方策が学習によって改善することを示す。
具体的な学習アルゴリズム（DQN/PPO等の本格実装）は対象外で、あくまで
「外部ライブラリなしでも動く」ことを示す最小例。

[▶ Google Colabで開く](https://colab.research.google.com/github/tmwork1/jpoke/blob/main/examples/04_others/rl_q_learning.ipynb)

In [ ]:
%pip install -q jpoke

In [ ]:
import random
from collections import defaultdict

from jpoke import Player
from jpoke.players import RandomPlayer
from jpoke.rl import ACTION_SPACE_SIZE, RLBattleEnv, RewardWeights, embed_battle_basic

random.seed(0)

## Qテーブル用のヘルパー

`embed_battle_basic()`（HP割合・瀕死・状態異常有無の最小観測ベクトル）を丸めて
Qテーブルのキーにする。行動選択は`get_action_mask()`相当の合法手マスク
（`RLBattleEnv.step()`/`reset()`が返す）の範囲でepsilon-greedyに行う。

In [ ]:
def state_key(env, player):
    """観測ベクトルを丸めてQテーブルのキーにする（最小の状態離散化）。"""
    return tuple(round(x, 1) for x in embed_battle_basic(env.battle, player))


def choose_action(Q, state, mask, epsilon):
    """epsilon-greedyで合法手の中から行動を選ぶ。"""
    legal = [a for a, ok in enumerate(mask) if ok]
    if random.random() < epsilon:
        return random.choice(legal)
    return max(legal, key=lambda a: Q[state][a])


def run_episode(env, Q, epsilon, alpha=0.3, gamma=0.9):
    """1エピソード分の対戦を実行し、Qテーブルを更新して勝敗を返す。"""
    mask, _ = env.reset()
    state = state_key(env, env.learner)
    terminated = truncated = False
    while not (terminated or truncated):
        action = choose_action(Q, state, mask, epsilon)
        mask, reward, terminated, truncated, _ = env.step(action)
        next_state = state_key(env, env.learner)
        legal_next = [a for a, ok in enumerate(mask) if ok]
        best_next = max((Q[next_state][a] for a in legal_next), default=0.0)
        Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])
        state = next_state
    return env.battle.won(env.learner)

## 対戦環境の準備

学習対象（`Learner`）は「威力の低いたいあたり」と「効果抜群の10まんボルト」の
2択を毎ターン迫られる。ランダムに選べば弱い方も同じ確率で選んでしまうが、
Q学習によって「10まんボルトを選び続ける」方策に収束することを確認する。

In [ ]:
learner = Player("Learner")
learner.add_pokemon("マルマイン", moves=["たいあたり", "10まんボルト"])

opponent = RandomPlayer("Opponent")
opponent.add_pokemon("ラプラス", moves=["なみのり"])

# HP割合・瀕死・勝敗を重み付けした報酬（勝敗のみのスパースな報酬より学習が早い）
weights = RewardWeights(hp=1.0, fainted=1.0, victory=5.0)
env = RLBattleEnv(learner, opponent, max_turns=20, n_selected=1, reward_weights=weights)
Q = defaultdict(lambda: [0.0] * ACTION_SPACE_SIZE)

In [ ]:
n_train_episodes = 200
wins = sum(run_episode(env, Q, epsilon=0.3) for _ in range(n_train_episodes))
print(f"学習中（epsilon=0.3）の勝率: {wins}/{n_train_episodes}")

In [ ]:
n_eval_episodes = 100
eval_wins = sum(run_episode(env, Q, epsilon=0.0) for _ in range(n_eval_episodes))
print(f"学習後（greedy方策）の勝率: {eval_wins}/{n_eval_episodes}")
print(f"学習で観測した状態数: {len(Q)}")

## 制約・対象外

- 瀕死交代など、ターン内で複数回発生し得る意思決定はRL行動として公開していない。
  `learner`/`opponent`いずれも、強制交代は既定の`choose_command()`（先頭の合法手）に
  フォールバックする
- `embed_battle_basic()`は「HP割合・瀕死・状態異常有無」だけを並べた最小参考実装。
  技相性や場の状態を含む本格的な特徴量設計は利用者に委ねる
- 自己対戦（マルチエージェント）は対象外。対戦相手は既存の`Player`サブクラス
  （本例では`RandomPlayer`）をそのまま使う